# CLEANING OF 'family_inc_exp.parquet'

Changes 
1. Stadardize the column names to camelCase 
2. Matches the city and provinces to the PSGC provinces and cities by creating separate column (city / province) 
3. Make the annual income/expenses in thousands (e.g 100000)
4. Create new column from annual income and expense as monthly 
5. Create a foreign key for the location

Inputs:
- bronze: family_inc_exp
- silver: dim_geography

Output Columns:

- geography: string
- families: integer
- avgIncome: decimal(9,0)
- avgExpenses: decimal(9,0)
- monthlyIncome: decimal(10,0)
- monthlyExpenses: decimal(10,0)
- geographyFk: string


In [0]:
bronze_folder = 'abfss://bronze@asterisktotle01.dfs.core.windows.net/PHHousing'
silver_folder = 'abfss://silver@asterisktotle01.dfs.core.windows.net/PHHousing'

df_fam = spark.read.format('parquet').load(bronze_folder+'/family_inc_exp')




In [0]:
from pyspark.sql import functions as F
import re
# Remove the 'city of' and 'city' from the 'city_province'
from pyspark.sql import functions as F

df_fam = df_fam.withColumn('city_province', F.trim(F.col('city_province'))) \
       .withColumn('city_province', F.regexp_replace(F.col('city_province'), r"(?i)^(City of)\s+", "")) \
       .withColumn('city_province', F.regexp_replace(F.col('city_province'), r"(?i)\s+city$", "")) \
       .withColumn('city_province', F.regexp_replace(F.col('city_province'), "±", "ñ")) \
       .withColumn('city_province', F.lower(F.col('city_province')))




In [0]:
df_fam = df_fam.withColumn('avg_inc', F.round(F.col('avg_inc') * 1000)).withColumn('avg_exp', F.round(F.col('avg_exp') * 1000))

df_fam = df_fam.withColumn('monthlyInc', F.round((F.col('avg_inc') / 12))).withColumn('monthlyExp',F.round(F.col('avg_exp') / 12))

df_fam = df_fam.withColumn('netIncome', F.round(F.col('monthlyInc') - F.col('monthlyExp')))




In [0]:
df_geography = spark.read.format('delta').load(silver_folder + '/dim_geography')
display(df_geography)

In [0]:
df_geography = spark.read.format('delta').load(silver_folder + '/dim_geography')

display(df_geography)

In [0]:
df_geography = spark.read.format('delta').load(silver_folder + '/dim_geography')


geo_city = (
    df_geography
    .filter(F.col('geographyLevel') == 'City')
    .select('geographyFk', 'geographyName') 
    .withColumnRenamed('geographyName', 'cityName')
    .withColumnRenamed('geographyFk', 'cityFk')

)
geo_province = (
    df_geography
    .filter(F.col('geographyLevel') == 'Province')
    .select('geographyFk', 'geographyName')
    .withColumnRenamed('geographyName', 'provinceName')

    .withColumnRenamed('geographyFk', 'provinceFk')
)

dm_fam_fk = (df_fam
             .join(geo_city, df_fam.city_province == geo_city.cityName, 'left')
             .join(geo_province, df_fam.city_province == geo_province.provinceName, 'left')
             .withColumn('geographyFk', F.coalesce(F.col('cityFk'), F.col('provinceFk')))
             .drop('cityFk', 'provinceFk', 'cityName', 'provinceName')
             )

dim_fam = (dm_fam_fk
           .withColumnsRenamed({'city_province': 'geography', 'total_families': 'families', 'avg_inc': 'avgIncome', 'avg_exp': 'avgExpenses', 'monthlyInc': 'monthlyIncome', 'monthlyExp': 'monthlyExpenses'}))



dim_fam.display()
dim_fam.write.mode('overwrite').format('delta').save(silver_folder + '/dim_fam')
display(dim_fam)    
print('dim_fam saved')